# Task 1: MongoDB Data Model

This notebook covers the MongoDB collection design and relationship justification for the AWAS (Automated Awareness Safety System) traffic monitoring pipeline. The system ingests real-time camera events across three fixed checkpoints, detects speed violations, and persists structured enforcement records.

---

## Task 1.1 — Collection Design

Three core collections are designed: **vehicles**, **cameras**, and **violations**. Each is described below with its purpose, schema, a representative sample document, index strategy, sharding consideration, and retention policy.

---

### 1. `vehicles` Collection

**Purpose:** Stores static master data for all registered vehicles. This collection acts as the reference dimension for ownership lookups during enforcement reporting. It is loaded once from `vehicle.csv` and updated only when registration details change.

**Schema:**

| Field | Type | Description |
|---|---|---|
| `_id` | ObjectId | MongoDB internal document ID |
| `car_plate` | String | Vehicle registration plate (unique natural key) |
| `owner_name` | String | Registered owner full name |
| `owner_addr` | String | Registered owner address |
| `vechicle_type` | String | Vehicle category (e.g., Sedan, SUV, Coupe) |
| `registration_date` | ISODate | Date and time of vehicle registration |

**Sample Document:**

```json
{
  "_id": ObjectId("..."),
  "car_plate": "DQZ 793",
  "owner_name": "Muhammad bin Liyana",
  "owner_addr": "946 Jalan Bukit Jelutong, Kuala Lumpur",
  "vechicle_type": "SUV",
  "registration_date": ISODate("2008-04-13T09:01:23")
}
```

**Indexes:**

```js
// Primary lookup index — car_plate is the join key for violation lookups
db.vehicles.createIndex({ "car_plate": 1 }, { unique: true })
```

**Shard Key Strategy:** Not sharded. The vehicles collection is small (~10,000 documents) and read-heavy with no write contention. A single replica set node handles this load comfortably. If the dataset scaled to millions of vehicles nationally, `car_plate` would be a reasonable hashed shard key for even distribution.

**Retention Policy:** Documents are retained indefinitely while the vehicle remains registered. Records for de-registered vehicles may be soft-deleted (adding a `deregistered_date` field) rather than hard-deleted, to preserve historical violation linkage.

---

### 2. `cameras` Collection

**Purpose:** Stores static metadata for each roadside camera checkpoint, including its geographic coordinates, road position marker, and enforced speed limit. Camera data is loaded from `camera.csv` at startup. Inter-camera distances are **not** stored; they are computed dynamically at query or stream-processing time using the Haversine formula applied to the `latitude` and `longitude` fields. This ensures correctness if camera positions are ever updated.

**Schema:**

| Field | Type | Description |
|---|---|---|
| `_id` | ObjectId | MongoDB internal document ID |
| `camera_id` | Integer | Unique camera identifier |
| `location` | GeoJSON Point | GeoJSON object storing longitude, latitude for geospatial indexing |
| `latitude` | Double | Raw latitude for Haversine distance computation |
| `longitude` | Double | Raw longitude for Haversine distance computation |
| `position` | Double | Road kilometre marker (km) |
| `speed_limit` | Integer | Enforced speed limit at this camera (km/h) |

**Sample Document:**

```json
{
  "_id": ObjectId("..."),
  "camera_id": 1,
  "location": {
    "type": "Point",
    "coordinates": [102.6601002, 2.157730731]
  },
  "latitude": 2.157730731,
  "longitude": 102.6601002,
  "position": 152.5,
  "speed_limit": 110
}
```

**Indexes:**

```js
// Primary lookup — camera_id is referenced in every violation record
db.cameras.createIndex({ "camera_id": 1 }, { unique: true })

// Geospatial index — enables $near queries and supports dynamic Haversine lookups
db.cameras.createIndex({ "location": "2dsphere" })
```

**Shard Key Strategy:** Not sharded. Only 3 camera documents exist in this dataset. The collection is effectively a configuration lookup table. At national scale (thousands of cameras), `camera_id` with range-based sharding would suffice.

**Retention Policy:** Camera records are retained for the operational lifetime of each device. When a camera is decommissioned or relocated, the old document is updated in-place rather than deleted, preserving historical speed limit and position context for past violation records that reference it.

---

### 3. `violations` Collection

**Purpose:** Persists all detected speed violations produced by the Spark Structured Streaming pipeline. The design uses a **daily aggregation pattern**: one document per vehicle per calendar date, with individual violation events embedded as an array. This consolidates multiple violations for the same car on the same day into a single document, minimising document proliferation and aligning with the assignment requirement to merge same-day records.

Two violation types are captured:
- `INSTANTANEOUS` — speed at a single camera exceeds that camera's limit
- `AVERAGE` — computed average speed between two consecutive cameras exceeds the ending camera's limit; distance is derived dynamically via Haversine

**Schema:**

| Field | Type | Description |
|---|---|---|
| `_id` | ObjectId | MongoDB internal document ID |
| `car_plate` | String | References `vehicles.car_plate` |
| `date` | String | Calendar date (YYYY-MM-DD) - daily partition key |
| `violations` | Array | Embedded array of individual violation event objects |
| `violations[].violation_type` | String | `INSTANTANEOUS` or `AVERAGE` |
| `violations[].camera_id_start` | Integer | Camera where the event or segment begins |
| `violations[].camera_id_end` | Integer | Camera where the event or segment ends (same as start for instantaneous) |
| `violations[].timestamp_start` | ISODate | Timestamp of the starting camera event |
| `violations[].timestamp_end` | ISODate | Timestamp of the ending camera event |
| `violations[].speed_reading` | Double | Recorded speed (instantaneous) or computed average speed (km/h) |

**Sample Document:**

```json
{
  "_id": ObjectId("..."),
  "car_plate": "CJW 924",
  "date": "2024-01-01",
  "violations": [
    {
      "violation_type": "INSTANTANEOUS",
      "camera_id_start": 1,
      "camera_id_end": 1,
      "timestamp_start": ISODate("2024-01-01T08:00:01Z"),
      "timestamp_end": ISODate("2024-01-01T08:00:01Z"),
      "speed_reading": 148.3
    },
    {
      "violation_type": "AVERAGE",
      "camera_id_start": 1,
      "camera_id_end": 2,
      "timestamp_start": ISODate("2024-01-01T08:00:01Z"),
      "timestamp_end": ISODate("2024-01-01T08:00:26Z"),
      "speed_reading": 144.0
    }
  ]
}
```

**Indexes:**

```js
// Compound unique index — enforces one document per vehicle per day; also serves as the upsert filter key
db.violations.createIndex({ "car_plate": 1, "date": 1 }, { unique: true })

// Date index — supports time-range queries for visualisation and reporting
db.violations.createIndex({ "date": 1 })
```

**Shard Key Strategy:** `{ car_plate: 1, date: 1 }` as a compound hashed shard key would distribute write load evenly across shards in a high-throughput enforcement system, since violations are written continuously for different vehicles. For this assignment's scale, sharding is not required.

**Retention Policy:** Violation records must be retained for the statutory traffic enforcement period. In Malaysia, the Road Transport Act 1987 supports administrative enforcement actions; a retention window of 7 years is a reasonable default. Records older than 7 years may be archived to cold storage (e.g., MongoDB Atlas Online Archive or S3) and purged from the live collection using a TTL index on `date`:

```js
// TTL index — automatically expires documents 7 years after the violation date
db.violations.createIndex(
  { "date": 1 },
  { expireAfterSeconds: 220752000 }  // 7 years in seconds
)
```
Note: This design supports Requirement 2.1.4 by merging multiple violations for the same vehicle on the same day into a single record. A unique compound index on { "car_plate": 1, "date": 1 } ensures each vehicle-date pair is stored once, while stream processing uses Upsert and $push operations to append new violations to the existing daily record without creating duplicates.

---

### 4. `camera_events_historic` Collection

**Purpose:** Stores pre-existing historical violation records loaded from `camera_event_historic.csv`. This collection serves as a static reference baseline for trend analysis and visualisation, providing enforcement context prior to the live streaming pipeline. It is loaded once at setup and is not written to by the stream.

**Schema:**

| Field | Type | Description |
|---|---|---|
| `_id` | ObjectId | MongoDB internal document ID |
| `violation_id` | String | UUID uniquely identifying the historical violation record |
| `car_plate` | String | Vehicle registration plate; references `vehicles.car_plate` |
| `camera_id_start` | Integer | Camera at the start of the monitored segment |
| `camera_id_end` | Integer | Camera at the end of the monitored segment |
| `timestamp_start` | ISODate | Timestamp when the vehicle passed the starting camera |
| `timestamp_end` | ISODate | Timestamp when the vehicle passed the ending camera |
| `speed_reading` | Double | Recorded or computed speed at time of violation (km/h) |

**Sample Document:**

```json
{
  "_id": ObjectId("..."),
  "violation_id": "0ff38c74-7ee6-41cd-bc26-3a6060604a02",
  "car_plate": "YE 6517",
  "camera_id_start": 1,
  "camera_id_end": 2,
  "timestamp_start": ISODate("2018-11-14T08:30:11Z"),
  "timestamp_end": ISODate("2018-11-14T08:30:35.821118Z"),
  "speed_reading": 145.0
}

// Lookup by vehicle plate — supports joining with vehicles collection for reporting
db.camera_events_historic.createIndex({ "car_plate": 1 })

// Segment-level queries — supports filtering violations by camera pair
db.camera_events_historic.createIndex({ "camera_id_start": 1, "camera_id_end": 1 })

// Unique constraint on violation_id — prevents duplicate loads on re-ingestion
db.camera_events_historic.createIndex({ "violation_id": 1 }, { unique: true })

Shard Key Strategy: Not sharded. This is a static, read-only reference dataset loaded once from CSV. No write contention exists. At larger historical scale, car_plate hashed sharding would distribute read load evenly.

Retention Policy: Permanent. This collection represents the historical record of past enforcement events and should not expire. No TTL index is applied.

## Task 1.2 — Collection Relationships

### Relationship Overview

The three collections relate to each other as follows:

```
vehicles (1) ──────────────────── (*) violations
                                        │
cameras  (1) ─── camera_id_start ──────┤
cameras  (1) ─── camera_id_end   ──────┘
```

- A **vehicle** can have zero or many **violation** daily records over its lifetime.
- Each **violation** document references exactly one vehicle via `car_plate`.
- Each violation event within the embedded array references one or two cameras via `camera_id_start` and `camera_id_end`.
- **cameras** and **vehicles** have no direct relationship; they are independent reference dimensions.

---

### Embed vs. Reference Decisions

#### Decision 1: Violation events are **embedded** inside the daily `violations` document

Individual violation events (instantaneous and average) are stored as an embedded array within the parent daily document, rather than as separate top-level documents.

**Justification:**

The primary read pattern for enforcement is to retrieve all violations for a specific vehicle on a specific day — a single document fetch using the `{ car_plate, date }` compound index. Embedding keeps this a single-document read with no joins. The write pattern from the Spark sink also favours embedding: new violation events are appended using $addToSet inside an upsert, which is atomic at the document level and idempotent against duplicate Spark batch replays.

The trade-off is document growth. MongoDB's 16 MB document size limit is not a concern here: even a vehicle that triggers hundreds of violations per day would produce a document well under 1 MB. Array unboundedness is therefore not a practical risk at this dataset's scale.

Note: The $push operator used during upsert writes is atomic at the document level, ensuring that concurrent Spark workers processing the same micro-batch cannot produce duplicate or corrupted daily records

#### Decision 2: Vehicle ownership data is **referenced** from violations (not embedded)

The `violations` documents store only `car_plate`, which acts as a foreign key reference to the `vehicles` collection. Owner name, address, and vehicle type are not duplicated into violation records.

**Justification:**

Vehicle registration details can change (e.g., ownership transfer, address update). If this data were embedded inside each violation document, every ownership change would require a costly multi-document update across all historical violation records for that plate. By referencing via `car_plate`, the `vehicles` collection remains the single source of truth, and any update propagates automatically to all downstream lookups.

The read cost of a join (application-level lookup from violations → vehicles) is acceptable because enforcement reporting is not a real-time path — it happens post-stream in the visualisation layer, where a single batch query can enrich violation records with vehicle details in one round trip.

#### Decision 3: Camera metadata is **referenced** from violations (not embedded)

Violation records store `camera_id_start` and `camera_id_end` as integer references, rather than embedding camera coordinates or speed limits directly.

**Justification:**

Camera configurations can change — a camera may be relocated or have its speed limit updated by road authorities. Embedding camera metadata into historical violation records would freeze stale values and create inconsistency. Referencing ensures that any camera update is immediately reflected in future queries without requiring backfills.

Additionally, inter-camera distance is not stored anywhere in the data model. It is computed dynamically at processing time using the **Haversine formula** applied to the `latitude` and `longitude` fields retrieved from the `cameras` collection. This ensures that if a camera is physically relocated and its coordinates are updated, all subsequent average speed calculations will use the correct distance automatically.

#Note: Although `camera.csv` provides linear road position markers, the model uses the Haversine formula with latitude and longitude to achieve more accurate distance calculations across curved or non-linear road segments and to remain adaptable for future road network changes.


---

### Summary Table

| Relationship | Strategy | Reason |
|---|---|---|
| Violation → individual violation events | **Embed** | Same-document read/write; bounded growth; atomic `$push` upserts |
| Violation → Vehicle | **Reference** (`car_plate`) | Ownership data changes; single source of truth; avoids costly backfills |
| Violation → Camera | **Reference** (`camera_id`) | Camera config/location may change; distances computed dynamically via Haversine |

### Technical Assumptions

To ensure the robustness of the stream processing logic and the integrity of the enforcement data, the following technical assumptions have been made:

| Assumption | Reasoning |
| :--- | :--- |
| **Event Reordering** | Kafka does not guarantee strict ordering across different partitions; Spark Structured Streaming's `eventTime` and watermarking are utilized to handle late or out-of-order arrivals. |
| **Missing Camera Matches** | If a vehicle is recorded at Camera A but fails to appear at Camera B within the stateful join window, the record is treated as unmatched and no average speed violation is raised. |
| **Valid Segment Direction** | Monitored segments are strictly A $\rightarrow$ B and B $\rightarrow$ C. Reverse travel or "skipping" (A $\rightarrow$ C) are treated as invalid sequences and excluded from average speed detection. |
| **Distance via Haversine** | Applied to `latitude` and `longitude` retrieved from the camera master data to maintain mathematical correctness across varied road geometries. |
| **7-Year Retention Period** | Aligns with administrative enforcement precedents and statutory limitations for document preservation under Malaysia's Road Transport Act 1987. |